# Tratamento dos Dados — Mortalidade 

---
## 1. Imports

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais


In [2]:
# ── Caminhos ─────────────────────────────────────────────────────────────────
PASTA  = Path("/home/anna/tcc/Limpeza e Integração/Mortalidade/Cidades Atendidas por Cada Regional")
SAIDA  = Path("/home/anna/tcc/Seleção e Transformação/Mortalidade")
ARQUIVO = SAIDA / "tabela_mortalidade_faixa_etaria.csv"

ARQUIVOS_MUNICIPIOS = [
    Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv"),
    Path("/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv"),
]

# ── Período de análise ────────────────────────────────────────────────────────
ANOS = list(range(1979, 2025))

# ── Faixas etárias padronizadas ───────────────────────────────────────────────
FAIXAS_ETARIAS = [
    "0-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70 OU MAIS",
]

# ── Filtro de sexo ────────────────────────────────────────────────────────────
# "mulheres" → mantém só arquivos de mulheres
# "homens"   → mantém só arquivos de homens
# None       → mantém todos os arquivos independentemente do sexo
SEXO_DESEJADO: str | None = "mulheres"

# ── CID-10 de interesse ───────────────────────────────────────────────────────
CID_PREFERIDO = "c50"   # C50 = neoplasia maligna da mama

# ── Padrões compilados (não altere) ──────────────────────────────────────────
PADRAO_ANO_ARQUIVO = re.compile(r"^(19\d{2}|20\d{2})")
PADRAO_INTERVALO   = re.compile(r"(\d{1,3})\s*(?:a|-|ate|até)\s*(\d{1,3})", re.IGNORECASE)
PADRAO_OU_MAIS     = re.compile(r"^(\d{1,3})\s*(?:ou\s*mais|\+|e\s*mais|99\+)$", re.IGNORECASE)

SAIDA.mkdir(parents=True, exist_ok=True)
print(f"Pasta de saída: {SAIDA}")
print(f"   Período: {ANOS[0]}–{ANOS[-1]} | Sexo: {SEXO_DESEJADO} | CID: {CID_PREFERIDO.upper()}")

Pasta de saída: /home/anna/tcc/Seleção e Transformação/Mortalidade
   Período: 1979–2024 | Sexo: mulheres | CID: C50


---
## 3. Funções de Normalização de Texto

Funções para padronizar rótulos de faixa etária, nomes de municípios e parsing de intervalos de idade.

In [3]:
def normalizar_texto(texto: str) -> str:
    """Remove acentos, converte para minúsculas e normaliza espaços e hífens."""
    txt = unicodedata.normalize("NFKD", str(texto).strip())
    txt = "".join(ch for ch in txt if not unicodedata.combining(ch))
    txt = txt.lower().replace("_", " ").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", txt).strip()


def remover_sufixo_mg(texto: str) -> str:
    """Remove ' - MG' e variações do final do nome do município."""
    txt = normalizar_texto(texto)
    txt = re.sub(r"\s*-\s*mg$", "", txt)
    txt = re.sub(r"\s+mg$", "", txt)
    return txt.strip()


def normalizar_chave_municipio(texto: str) -> str:
    """
    Normalização completa para uso como chave de join com a tabela de municípios.
    Remove sufixo MG, substitui hífens e apóstrofos por espaço e resolve aliases.
    """
    txt = remover_sufixo_mg(texto)
    txt = txt.replace("-", " ").replace("'", " ")
    txt = re.sub(r"\s+", " ", txt).strip()
    aliases = {
        "dona eusebia"         : "dona euzebia",
        "sao thome das letras" : "sao tome das letras",
        "brasopolis"           : "brazopolis",
    }
    return aliases.get(txt, txt)


---
## 4. Funções de Parsing de Faixa Etária

Converte rótulos de coluna dos CSVs (ex: "60 a 64", "70 ou mais") para as 8 faixas padronizadas.

In [4]:
def _faixa_por_idade_inicial(ini: int) -> str | None:
    """Retorna a faixa padronizada a partir da idade inicial de um intervalo."""
    if ini < 0:  return None
    if ini < 10: return "0-9"
    if ini < 20: return "10-19"
    if ini < 30: return "20-29"
    if ini < 40: return "30-39"
    if ini < 50: return "40-49"
    if ini < 60: return "50-59"
    if ini < 70: return "60-69"
    return "70 OU MAIS"


def _parse_intervalo_idade(rotulo: str) -> tuple[int, int] | None:
    """
    Extrai (idade_inicial, idade_final) de um rótulo de coluna.
    Retorna None para rótulos não numéricos (total, CID, ignorado etc.).
    """
    t = normalizar_texto(rotulo)
    IGNORAR = {
        "cid", "ci", "total", "ignorado", "ign", "obito", "óbito",
        "obitos", "óbitos", "subtotal", "sem informacao", "sem informação",
        "ign.", "ign",
    }
    if not t or t in IGNORAR or not t[0].isdigit():
        return None

    m_mais = PADRAO_OU_MAIS.search(t)
    if m_mais:
        return (int(m_mais.group(1)), 999)

    m_int = PADRAO_INTERVALO.search(t)
    if m_int:
        ini, fim = int(m_int.group(1)), int(m_int.group(2))
        return (ini, 999 if fim >= 99 else fim)

    if t.isdigit():
        idade = int(t)
        return (idade, idade)

    return None


def mapear_faixa_etaria(rotulo: str) -> str | None:
    """
    Mapeia um rótulo bruto de coluna para a faixa etária padronizada.
    Rejeita intervalos com mais de 10 anos (rótulos de subtotal).
    """
    intervalo = _parse_intervalo_idade(rotulo)
    if intervalo is None:
        return None
    ini, fim = intervalo
    if fim < ini:
        return None
    if fim != 999 and (fim - ini) > 10:
        return None
    faixa = _faixa_por_idade_inicial(ini)
    return faixa if faixa in set(FAIXAS_ETARIAS) else None


---
## 5. Funções de Leitura dos CSVs Brutos

Os CSVs do DATASUS/SIM têm estrutura não padronizada: linhas de metadados no topo,
cabeçalho iniciando com "CID;" e valores numéricos separados por ";".
As funções abaixo localizam automaticamente o cabeçalho e a linha de dados do CID de interesse.

In [5]:
def ler_linhas_csv(csv_path: Path) -> list[str]:
    """Lê as linhas de um CSV tentando latin-1 primeiro, depois utf-8."""
    for enc in ("latin-1", "utf-8"):
        try:
            with open(csv_path, "r", encoding=enc, errors="replace") as f:
                return [ln.rstrip("\n") for ln in f]
        except Exception:
            continue
    return []


def detectar_sexo_arquivo(csv_path: Path) -> str | None:
    """Detecta o sexo referenciado no arquivo lendo as 3 primeiras linhas."""
    linhas = ler_linhas_csv(csv_path)
    topo   = " ".join(normalizar_texto(ln) for ln in linhas[:3] if ln.strip())
    if "em homens"   in topo: return "homens"
    if "em mulheres" in topo: return "mulheres"
    return None


def split_csv_line(linha: str) -> list[str]:
    """Divide linha CSV por ';' e remove células vazias do final."""
    partes = [c.strip() for c in str(linha).split(";")]
    while partes and partes[-1] == "":
        partes.pop()
    return partes


def extrair_pares_cabecalho_valor(csv_path: Path, cid_preferido: str = CID_PREFERIDO) -> list[tuple[str, str]]:
    """
    Localiza a linha de cabeçalho (começa com 'CID;') e a linha de dados
    correspondente ao CID de interesse.
    Retorna lista de (rótulo_coluna, valor) para uso no mapeamento de faixas.
    """
    linhas  = ler_linhas_csv(csv_path)
    idx_cab = next(
        (i for i, ln in enumerate(linhas) if re.match(r"^\s*CID\s*;", ln, re.IGNORECASE)),
        None,
    )
    if idx_cab is None or idx_cab + 1 >= len(linhas):
        return []

    cabecalho  = split_csv_line(linhas[idx_cab])
    candidatos = [
        ln for ln in linhas[idx_cab + 1:]
        if ";" in ln and len(split_csv_line(ln)) >= 2
        and any(p.strip() for p in split_csv_line(ln)[1:])
    ]
    if not candidatos:
        return []

    # Prefere a linha que menciona o CID de interesse; usa a primeira caso contrário
    linha_dados = next(
        (
            ln for ln in candidatos
            if re.search(rf"\b{re.escape(cid_preferido.lower())}\b",
                         normalizar_texto(split_csv_line(ln)[0]))
        ),
        candidatos[0],
    )

    valores = split_csv_line(linha_dados)
    return [(cabecalho[i], valores[i] if i < len(valores) else "") for i in range(len(cabecalho))]


def para_inteiro(valor) -> int | None:
    """Converte string numérica (com ponto ou vírgula) para int. Retorna None se inválido ou zero."""
    if valor is None:
        return None
    txt = str(valor).strip().replace(".", "").replace(",", "")
    if not txt or not txt.isdigit():
        return None
    n = int(txt)
    return n if n > 0 else None


---
## 6. Consolidação da Mortalidade

`consolidar_mortalidade` percorre a árvore de diretórios `regional → município → CSVs`,
lê cada arquivo anual, extrai os pares (faixa etária, valor) e acumula por município.

In [6]:
def consolidar_mortalidade(pasta_raiz: Path) -> pd.DataFrame:
    """
    Percorre pasta_raiz/regional/municipio/*.csv e consolida os dados de mortalidade
    em um DataFrame com uma linha por município e colunas YYYY_faixa.
    Respeita o filtro SEXO_DESEJADO definido nas configurações.
    """
    registros = []

    for regional_dir in sorted(p for p in pasta_raiz.iterdir() if p.is_dir()):
        regional = regional_dir.name

        for municipio_dir in sorted(p for p in regional_dir.iterdir() if p.is_dir()):
            municipio = municipio_dir.name
            registro: dict[str, object] = {
                "Município"         : municipio,
                "Regional de Saúde" : regional,
            }

            for csv_path in sorted(municipio_dir.glob("*.csv")):
                # Filtro de sexo
                if SEXO_DESEJADO is not None:
                    sexo = detectar_sexo_arquivo(csv_path)
                    if sexo is not None and sexo != SEXO_DESEJADO:
                        continue

                ano = None
                m = PADRAO_ANO_ARQUIVO.search(csv_path.name)
                if m:
                    ano_cand = int(m.group(1))
                    ano = ano_cand if ano_cand in ANOS else None
                if ano is None:
                    continue

                pares = extrair_pares_cabecalho_valor(csv_path)
                for rotulo, valor_raw in pares:
                    faixa = mapear_faixa_etaria(rotulo)
                    if faixa is None:
                        continue
                    valor = para_inteiro(valor_raw)
                    if valor is None:
                        continue
                    coluna = f"{ano}_{faixa}"
                    anterior = registro.get(coluna)
                    registro[coluna] = valor if anterior is None else int(anterior) + valor

            registros.append(registro)

    return pd.DataFrame(registros)


---
## 7. Organização de Colunas e Enriquecimento com Código IBGE

`organizar_colunas` garante que todas as combinações `YYYY_faixa` existam (com `NaN` onde não há dado),
calcula `Total_YYYY` por ano (renomeado para `YYYY_TOTAL` na etapa de salvar) e reordena as colunas
na ordem canônica.

`carregar_mapa_codigos` e `adicionar_codigo_municipio` fazem o join com a tabela de municípios
para adicionar o código IBGE de 7 dígitos.

In [7]:
def organizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Garante que todas as colunas YYYY_faixa existam (NaN onde ausente)
    2. Converte para Int64 (suporta NaN)
    3. Calcula Total_YYYY por ano (renomeado para YYYY_TOTAL na etapa de salvar)
    4. Reordena: Município, Regional, colunas_ano_faixa, Total_YYYY
    """
    colunas_ano_faixa = [f"{ano}_{faixa}" for ano in ANOS for faixa in FAIXAS_ETARIAS]
    colunas_total_ano = [f"Total_{ano}" for ano in ANOS]

    # Cria colunas faltantes com NaN
    faltantes = [c for c in colunas_ano_faixa if c not in df.columns]
    if faltantes:
        df = pd.concat(
            [df, pd.DataFrame(np.nan, index=df.index, columns=faltantes)],
            axis=1,
        )

    for col in colunas_ano_faixa:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    for ano in ANOS:
        cols = [f"{ano}_{faixa}" for faixa in FAIXAS_ETARIAS]
        df[f"Total_{ano}"] = df[cols].sum(axis=1, min_count=1).astype("Int64")

    colunas_fixas  = ["Município", "Regional de Saúde"]
    extras         = [c for c in df.columns if c not in {*colunas_fixas, *colunas_ano_faixa, *colunas_total_ano}]
    colunas_finais = colunas_fixas + colunas_ano_faixa + extras + colunas_total_ano

    return (
        df[colunas_finais]
        .sort_values(["Regional de Saúde", "Município"], na_position="last")
        .reset_index(drop=True)
    )


def carregar_mapa_codigos() -> dict[str, str]:
    """
    Carrega o mapeamento MUNICÍPIO NORMALIZADO → CÓDIGO IBGE do primeiro arquivo
    de ARQUIVOS_MUNICIPIOS que existir.
    """
    arquivo = next((p for p in ARQUIVOS_MUNICIPIOS if p.exists()), None)
    if arquivo is None:
        candidatos = ", ".join(str(p) for p in ARQUIVOS_MUNICIPIOS)
        raise FileNotFoundError(f"Arquivo de municípios não encontrado. Tentativas: {candidatos}")

    ref = pd.read_csv(arquivo, dtype={"CÓDIGO IBGE": "string"})
    obrigatorias = {"MUNICÍPIO NORMALIZADO", "CÓDIGO IBGE"}
    faltantes    = obrigatorias - set(ref.columns)
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes: {sorted(faltantes)}")

    ref = ref.dropna(subset=["MUNICÍPIO NORMALIZADO", "CÓDIGO IBGE"]).copy()
    ref["MUNICÍPIO NORMALIZADO"] = ref["MUNICÍPIO NORMALIZADO"].astype(str).map(normalizar_chave_municipio)
    ref["CÓDIGO IBGE"]           = ref["CÓDIGO IBGE"].astype(str).str.strip()
    ref = ref.drop_duplicates(subset=["MUNICÍPIO NORMALIZADO"], keep="first")

    return dict(zip(ref["MUNICÍPIO NORMALIZADO"], ref["CÓDIGO IBGE"]))


def adicionar_codigo_municipio(df: pd.DataFrame, mapa_codigos: dict[str, str]) -> pd.DataFrame:
    """
    Faz join pelo nome normalizado do município e insere a coluna 'Código'
    logo após 'Município'. Lista municípios sem correspondência para revisão.
    """
    if "Município" not in df.columns:
        raise ValueError("DataFrame não possui a coluna 'Município'.")

    chaves  = df["Município"].astype(str).map(normalizar_chave_municipio)
    codigos = chaves.map(mapa_codigos).astype("string")

    sem_codigo = (
        df.loc[codigos.isna(), "Município"].astype(str).dropna().sort_values().unique().tolist()
    )
    if sem_codigo:
        print(f"⚠️  {len(sem_codigo)} município(s) sem correspondência de código:")
        for mun in sem_codigo:
            print(f"   - {mun}")
    else:
        print("✅ Todos os municípios tiveram código IBGE encontrado.")

    df = df.drop(columns=["CÓDIGO", "Código"], errors="ignore").copy()
    pos = list(df.columns).index("Município") + 1
    df.insert(pos, "Código", codigos)
    return df

---
## 8. Pipeline de Tratamento


In [8]:
print("Verificando estrutura de diretórios...")
regionais = [p for p in PASTA.iterdir() if p.is_dir()]
municipios_total = sum(1 for r in regionais for p in r.iterdir() if p.is_dir())
print(f"   Regionais encontradas : {len(regionais)}")
print(f"   Municípios encontrados: {municipios_total}")

Verificando estrutura de diretórios...
   Regionais encontradas : 29
   Municípios encontrados: 853


In [9]:
# ── Etapa 1: Consolidação ────────────────────────────────────────────────────
print("\nEtapa 1: Consolidando mortalidade...")
df = consolidar_mortalidade(PASTA)
print(f"   Shape após consolidação: {df.shape}")


Etapa 1: Consolidando mortalidade...


   Shape após consolidação: (853, 290)


In [10]:
# ── Etapa 2: Organização de colunas e totais ────────────────────────────────
print("\nEtapa 2: Organizando colunas e calculando totais...")
df = organizar_colunas(df)
print(f"   Shape após organização: {df.shape}")


Etapa 2: Organizando colunas e calculando totais...
   Shape após organização: (853, 416)


In [11]:
# ── Etapa 3: Normalização e código IBGE ─────────────────────────────────────
print("\nEtapa 3: Adicionando código IBGE...")
df["Município"] = df["Município"].map(remover_sufixo_mg)
mapa_codigos    = carregar_mapa_codigos()
df              = adicionar_codigo_municipio(df, mapa_codigos)


Etapa 3: Adicionando código IBGE...
✅ Todos os municípios tiveram código IBGE encontrado.


In [12]:
# ── Etapa 4: Salvar ──────────────────────────────────────────────────────────
print("\n💾 Etapa 4: Salvando CSV...")

# Padroniza os nomes finais de colunas: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, YYYY_TOTAL
rename_final = {
    "Município"         : "MUNICÍPIO",
    "Código"            : "CÓDIGO IBGE",
    "Regional de Saúde" : "REGIONAL DE SAÚDE",
}
rename_final.update({f"Total_{ano}": f"{ano}_TOTAL" for ano in ANOS})
df = df.rename(columns=rename_final)
df.to_csv(ARQUIVO, sep=",", index=False, encoding="utf-8")

colunas_total = [c for c in df.columns if c.endswith("_TOTAL")]
soma_total    = int(df[colunas_total].sum(skipna=True).sum())

print("\n╔══════════════════════════════════════════════════════╗")
print("║              Arquivo gerado com sucesso              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Arquivo  : {str(ARQUIVO.name):<41}║")
print(f"║  Linhas   : {len(df):<41}║")
print(f"║  Colunas  : {len(df.columns):<41}║")
print(f"║  Municípios únicos  : {df['MUNICÍPIO'].nunique():<30}║")
print(f"║  Regionais únicas   : {df['REGIONAL DE SAÚDE'].nunique(dropna=True):<30}║")
print(f"║  Soma de todas YYYY_TOTAL : {soma_total:<24}║")
print("╚══════════════════════════════════════════════════════╝")


💾 Etapa 4: Salvando CSV...



╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Arquivo  : tabela_mortalidade_faixa_etaria.csv      ║
║  Linhas   : 853                                      ║
║  Colunas  : 417                                      ║
║  Municípios únicos  : 853                           ║
║  Regionais únicas   : 28                            ║
║  Soma de todas YYYY_TOTAL : 42422                   ║
╚══════════════════════════════════════════════════════╝


---
## 9. Inspeção Pós-Processamento

Verificação da tabela final: shape, amostra, valores ausentes e estatísticas do total.

In [13]:
display(Markdown(f"**Shape final:** {df.shape[0]} linhas × {df.shape[1]} colunas"))

colunas_total = [c for c in df.columns if c.endswith("_TOTAL")]
cols_preview  = ["MUNICÍPIO", "CÓDIGO IBGE", "REGIONAL DE SAÚDE"] + colunas_total[-5:]
display(Markdown("**Amostra (últimas colunas de total):**"))
display(df[cols_preview].head(5))

# Estatísticas do total geral (soma de todas as colunas YYYY_TOTAL por município)
total_por_municipio = df[colunas_total].sum(axis=1, skipna=True)
total_geral = int(total_por_municipio.sum())
media       = total_por_municipio.mean()
mediana     = total_por_municipio.median()
desvio      = total_por_municipio.std()
print(f"\nTotal de mortes no período : {total_geral:,}".replace(",", "."))
print(f"Média por município        : {media:,.1f}".replace(",", "."))
print(f"Mediana por município      : {mediana:,.1f}".replace(",", "."))
print(f"Desvio padrão              : {desvio:,.1f}".replace(",", "."))

# Municípios sem código
sem_cod = df["CÓDIGO IBGE"].isna().sum()
if sem_cod > 0:
    display(Markdown(f"⚠️ **{sem_cod} município(s) sem código IBGE:**"))
    display(df.loc[df["CÓDIGO IBGE"].isna(), ["MUNICÍPIO", "REGIONAL DE SAÚDE"]].head(10))
else:
    display(Markdown("✅ **Todos os municípios possuem código IBGE.**"))

**Shape final:** 853 linhas × 417 colunas

**Amostra (últimas colunas de total):**

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL
0,alfenas,3101607,Alfenas,7,9,6,11,7
1,alterosa,3102001,Alfenas,2,2,1,4,1
2,arceburgo,3104106,Alfenas,1,1,<NA>,2,<NA>
3,areado,3104304,Alfenas,1,1,<NA>,1,1
4,bandeira do sul,3105301,Alfenas,<NA>,2,1,<NA>,<NA>



Total de mortes no período : 42.422
Média por município        : 49.7
Mediana por município      : 11.0
Desvio padrão              : 331.9


✅ **Todos os municípios possuem código IBGE.**